# IPL Performance & Match Analytics (2008–2025)

**Objective:** Analyze IPL team performance, player batting and bowling, match outcomes, toss impact, venues, and scoring phases.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

DATA = Path("../Dataset")
matches = pd.read_csv(DATA / "matches_cleaned.csv", parse_dates=["date"])
deliveries = pd.read_csv(DATA / "deliveries_cleaned.csv", parse_dates=["date"])

print("Matches:", matches.shape)
print("Deliveries:", deliveries.shape)
display(matches.head())
display(deliveries.head())

## 1. Tournament KPIs

In [ ]:
total_matches = matches["match_id"].nunique()
decided_matches = matches["winner"].notna().sum()
total_runs = int(deliveries["total_runs"].sum())
legal_balls = int(deliveries["is_legal_ball"].sum())
print(f"Matches: {total_matches:,}")
print(f"Seasons: {matches['season'].nunique()}")
print(f"Recorded teams/franchise names: {pd.concat([matches.team1, matches.team2]).nunique()}")
print(f"Matches with a recorded winner: {decided_matches:,}")
print(f"Total runs in delivery data: {total_runs:,}")
print(f"Legal deliveries: {legal_balls:,}")

## 2. Team performance

In [ ]:
appearances = pd.concat([
    matches[["match_id","team1"]].rename(columns={"team1":"team"}),
    matches[["match_id","team2"]].rename(columns={"team2":"team"})
])
played = appearances.groupby("team")["match_id"].nunique()
wins = matches.dropna(subset=["winner"]).groupby("winner")["match_id"].nunique()
team_stats = pd.DataFrame({"matches_played":played, "wins":wins}).fillna(0)
team_stats["win_pct"] = (100*team_stats["wins"]/team_stats["matches_played"]).round(2)
team_stats = team_stats.sort_values(["wins","win_pct"],ascending=False)
display(team_stats.head(15))
ax = team_stats.head(10).sort_values("wins")["wins"].plot(kind="barh",figsize=(9,5),title="Top teams by recorded wins")
ax.set_xlabel("Wins"); ax.set_ylabel(""); plt.tight_layout(); plt.show()

## 3. Player batting analysis

In [ ]:
batting = deliveries.groupby("batsman").agg(
    runs=("batsman_runs","sum"),
    matches=("match_id","nunique")
)
balls_faced = deliveries.loc[deliveries["is_wide_flag"].eq(0)].groupby("batsman").size()
dismissals = deliveries.loc[deliveries["player_dismissed"].notna()].groupby("player_dismissed").size()
batting["balls_faced"] = balls_faced
batting["dismissals"] = dismissals
batting[["balls_faced","dismissals"]] = batting[["balls_faced","dismissals"]].fillna(0)
batting["strike_rate"] = (100*batting["runs"]/batting["balls_faced"].replace(0,np.nan)).round(2)
batting = batting.sort_values("runs",ascending=False)
display(batting.head(15))
batting.head(10).sort_values("runs")["runs"].plot(kind="barh",figsize=(9,5),title="Top run scorers")
plt.xlabel("Runs"); plt.ylabel(""); plt.tight_layout(); plt.show()

## 4. Player bowling analysis

In [ ]:
bowling = deliveries.groupby("bowler").agg(
    legal_balls=("is_legal_ball","sum"),
    runs_conceded=("bowler_runs_conceded","sum"),
    wickets=("bowler_wicket","sum"),
    matches=("match_id","nunique")
)
bowling["overs"] = bowling["legal_balls"]/6
bowling["economy"] = (bowling["runs_conceded"]/bowling["overs"].replace(0,np.nan)).round(2)
bowling = bowling.sort_values(["wickets","economy"],ascending=[False,True])
display(bowling.head(15))
bowling.head(10).sort_values("wickets")["wickets"].plot(kind="barh",figsize=(9,5),title="Top wicket takers")
plt.xlabel("Bowler-credited wickets"); plt.ylabel(""); plt.tight_layout(); plt.show()

## 5. Innings and scoring phases

In [ ]:
standard = deliveries[deliveries["inning"].isin([1,2])].copy()
innings = standard.groupby(["match_id","inning","batting_team","bowling_team"]).agg(
    runs=("total_runs","sum"), wickets=("is_wicket","sum"),
    legal_balls=("is_legal_ball","sum"), fours=("is_four","sum"), sixes=("is_six","sum")
).reset_index()
display(innings.head())
phase_stats = standard.groupby("over_phase").agg(
    runs=("total_runs","sum"), legal_balls=("is_legal_ball","sum"),
    fours=("is_four","sum"), sixes=("is_six","sum"), wickets=("is_wicket","sum")
)
phase_stats["run_rate"] = (phase_stats["runs"]/(phase_stats["legal_balls"]/6)).round(2)
display(phase_stats.reindex(["Powerplay","Middle","Death"]))
phase_stats["run_rate"].reindex(["Powerplay","Middle","Death"]).plot(kind="bar",figsize=(7,4),title="Run rate by innings phase")
plt.ylabel("Runs per over"); plt.xlabel(""); plt.xticks(rotation=0); plt.tight_layout(); plt.show()

## 6. Toss analysis

In [ ]:
decided = matches.dropna(subset=["winner","toss_winner"]).copy()
decided["toss_winner_won_match"] = decided["winner"].eq(decided["toss_winner"])
toss_stats = decided.groupby("toss_decision").agg(
    matches=("match_id","nunique"),
    toss_winner_match_wins=("toss_winner_won_match","sum")
)
toss_stats["win_pct"] = (100*toss_stats["toss_winner_match_wins"]/toss_stats["matches"]).round(2)
display(toss_stats)
print("Overall toss winner also won match:",
      f"{100*decided['toss_winner_won_match'].mean():.2f}%")

## 7. Venue and season analysis

In [ ]:
venue_stats = matches.groupby("venue").agg(
    matches=("match_id","nunique"),
    teams_winning_first=("winner_runs",lambda s:s.notna().sum())
).sort_values("matches",ascending=False)
season_stats = matches.groupby("season").agg(
    matches=("match_id","nunique"),
    decided_matches=("winner",lambda s:s.notna().sum())
).reset_index()
display(venue_stats.head(15))
display(season_stats)

## 8. Export analytical tables
The source and cleaned datasets are retained separately. These exports can be used in Power BI or PostgreSQL. Player names and team names are source labels; franchise rebrands are not consolidated automatically.

In [ ]:
OUT = Path("../Dataset")
team_stats.to_csv(OUT/"team_summary.csv")
batting.to_csv(OUT/"player_batting_summary.csv")
bowling.to_csv(OUT/"player_bowling_summary.csv")
innings.to_csv(OUT/"innings_summary.csv",index=False)
phase_stats.to_csv(OUT/"phase_summary.csv")
toss_stats.to_csv(OUT/"toss_analysis.csv")
season_stats.to_csv(OUT/"season_summary.csv",index=False)
venue_stats.to_csv(OUT/"venue_summary.csv")
print("Analytical exports saved to Dataset/.")

## Interpretation and limitations
- The match file contains 23 matches without a recorded winner: 15 ties and 8 no-results according to the `outcome` field.
- Three exact duplicate delivery rows were removed. No other delivery records were removed.
- Team names are preserved as recorded; rebrands such as Delhi Daredevils/Delhi Capitals and Kings XI Punjab/Punjab Kings are not automatically combined.
- Bowling wickets exclude run outs, retired hurt, obstructing the field, and retired out. Validate dismissal-credit rules before comparing with official leaderboards.
- Balls faced exclude wides; no-balls count as a ball faced. Legal balls exclude wides and no-balls.
- Standard innings analysis uses innings 1 and 2; super-over innings are excluded from innings-level score comparisons.
- Toss correlation is descriptive, not evidence that toss decisions cause match outcomes.
.